# LSTM Train Model

In [1]:
stock_code = "GAS"

In [2]:
output_column = "close"

In [3]:
stock_code = str.lower(stock_code)

## Install libraries

In [4]:
# !pip install -r requirements.txt

## Import libraries

In [5]:
import os
import sys

# go one level up from where the notebook is running
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from data_preprocessor.data_preprocessor import DataPreprocessor
from dtos.model_dtos.model_config_dto import ModelConfigDto
from logger.logger import Logger, LogType
from model.lstm_model import LSTM_Model
from train_test_creator.train_test_creator import TrainTestCreator
from utils.constants import LOG_FILE_BASE
from web_scraper.web_scraper import WebScraper
from utils.enums import *

## Load dataframe

In [6]:
my_logger = Logger(file_name=LOG_FILE_BASE)

In [7]:
my_train_test_creator = TrainTestCreator(logger=my_logger)

In [8]:
dataframe = my_train_test_creator.load_dataframe(
    stock_code=stock_code, file_path=f"../../unified_dataframe/unified_{stock_code}.csv"
)
normalized_df = my_train_test_creator.normalize_unified_dataframe(dataframe=dataframe)

In [9]:
normalized_df

,gdp_gdp_growth,xpi_animal_feed_and_raw_materials,xpi_aquatic_products,xpi_cameras_camcorders_and_components,xpi_cashew_nuts,xpi_cassava_and_cassava_products,xpi_chemical_products,xpi_chemicals,xpi_clinker_and_cement,xpi_coffee,...,is_quarter_end,week_of_month,half_of_year,season,month_sin,month_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos
0,0.517953,0.000000,0.312368,0.113804,0.172615,0.728733,0.051864,0.035475,0.000000,0.424677,...,0.0,0.50,0.0,0.333333,0.75,0.066987,0.307979,1.000000e+00,0.821030,0.116654
1,0.518058,0.000152,0.314136,0.113893,0.174513,0.724246,0.051997,0.035636,0.000099,0.427948,...,0.0,0.75,0.0,0.333333,0.75,0.066987,0.862937,8.019377e-01,0.814384,0.111185
2,0.518163,0.000304,0.315905,0.113983,0.176411,0.719759,0.052130,0.035797,0.000197,0.431220,...,0.0,0.75,0.0,0.333333,0.75,0.066987,1.000000,3.568959e-01,0.807645,0.105831
3,0.518268,0.000456,0.317673,0.114073,0.178310,0.715272,0.052263,0.035958,0.000296,0.434491,...,0.0,0.75,0.0,0.333333,0.75,0.066987,0.615957,5.840301e-17,0.800815,0.100594
4,0.518373,0.000608,0.319441,0.114163,0.180208,0.710784,0.052396,0.036119,0.000395,0.437763,...,0.0,0.75,0.0,0.333333,0.75,0.066987,0.000000,0.000000e+00,0.793895,0.095475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3264,0.706493,0.828807,0.800973,0.477067,0.878712,0.355854,0.493779,0.629226,0.347248,0.241666,...,0.0,0.50,0.0,0.666667,0.50,0.000000,0.000000,0.000000e+00,0.598337,0.009747
3265,0.708204,0.825273,0.789908,0.484927,0.859986,0.334605,0.492291,0.636682,0.339708,0.233438,...,0.0,0.75,0.0,0.666667,0.50,0.000000,0.307979,1.000000e+00,0.572900,0.005324
3266,0.709345,0.822916,0.782532,0.490167,0.847502,0.320439,0.491298,0.641653,0.334682,0.227953,...,0.0,0.75,0.0,0.666667,0.50,0.000000,1.000000,3.568959e-01,0.555830,0.003108
3267,0.709916,0.821738,0.778843,0.492787,0.841260,0.313356,0.490802,0.644138,0.332169,0.225211,...,0.0,0.75,0.0,0.666667,0.50,0.000000,0.615957,5.840301e-17,0.547269,0.002221


In [10]:
normalized_df.shape

(3269, 221)

## Create train test set

In [11]:
train_test_set = my_train_test_creator.create_train_test_set(
    normalized_df=normalized_df,
    output_column=output_column,
    stock_code=stock_code,
    input_window_size=DEFAULT_INPUT_WINDOW_SIZE,
    forecast_horizon_size=DEFAULT_FORECAST_HORIZON_SIZE,
)

In [12]:
print(f"Number of training samples: {train_test_set.get_number_of_train_windows()}")

Number of training samples: 86


In [13]:
print(
    f"Number of test forecast horizons: {train_test_set.get_number_of_test_forecast_horizons()}"
)

Number of test forecast horizons: 10


In [14]:
train_test_set.get_train_window()

,gdp_gdp_growth,xpi_animal_feed_and_raw_materials,xpi_aquatic_products,xpi_cameras_camcorders_and_components,xpi_cashew_nuts,xpi_cassava_and_cassava_products,xpi_chemical_products,xpi_chemicals,xpi_clinker_and_cement,xpi_coffee,...,week_of_month,half_of_year,season,month_sin,month_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos,close
0,0.517953,0.000000,0.312368,0.113804,0.172615,0.728733,0.051864,0.035475,0.000000,0.424677,...,0.50,0.0,0.333333,0.75,0.066987,0.307979,1.000000e+00,0.821030,0.116654,0.029092
1,0.518058,0.000152,0.314136,0.113893,0.174513,0.724246,0.051997,0.035636,0.000099,0.427948,...,0.75,0.0,0.333333,0.75,0.066987,0.862937,8.019377e-01,0.814384,0.111185,0.039867
2,0.518163,0.000304,0.315905,0.113983,0.176411,0.719759,0.052130,0.035797,0.000197,0.431220,...,0.75,0.0,0.333333,0.75,0.066987,1.000000,3.568959e-01,0.807645,0.105831,0.028553
3,0.518268,0.000456,0.317673,0.114073,0.178310,0.715272,0.052263,0.035958,0.000296,0.434491,...,0.75,0.0,0.333333,0.75,0.066987,0.615957,5.840301e-17,0.800815,0.100594,0.017778
4,0.518373,0.000608,0.319441,0.114163,0.180208,0.710784,0.052396,0.036119,0.000395,0.437763,...,0.75,0.0,0.333333,0.75,0.066987,0.000000,0.000000e+00,0.793895,0.095475,0.018317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385,0.606384,0.084511,0.474562,0.163705,0.222460,0.179463,0.061734,0.076417,0.054842,0.099805,...,0.75,1.0,1.000000,0.25,0.933013,0.615957,5.840301e-17,0.230995,0.921469,0.188931
386,0.606658,0.084663,0.470086,0.163795,0.222148,0.180214,0.061511,0.075568,0.054941,0.101645,...,1.00,1.0,1.000000,0.25,0.933013,0.000000,0.000000e+00,0.238290,0.926037,0.188931
387,0.607479,0.085119,0.465717,0.164064,0.220830,0.187289,0.061969,0.080935,0.055237,0.113725,...,0.00,1.0,0.000000,0.50,1.000000,0.307979,1.000000e+00,0.260631,0.938979,0.194879
388,0.607753,0.085271,0.465771,0.164154,0.220327,0.190451,0.062309,0.084043,0.055336,0.118845,...,0.00,1.0,0.000000,0.50,1.000000,0.862937,8.019377e-01,0.268223,0.943034,0.194879


In [15]:
train_test_set.get_train_window().shape

(390, 221)

In [16]:
train_test_set.get_test_window()

,gdp_gdp_growth,xpi_animal_feed_and_raw_materials,xpi_aquatic_products,xpi_cameras_camcorders_and_components,xpi_cashew_nuts,xpi_cassava_and_cassava_products,xpi_chemical_products,xpi_chemicals,xpi_clinker_and_cement,xpi_coffee,...,week_of_month,half_of_year,season,month_sin,month_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos,close
0,0.622690,0.651616,0.436154,0.660920,0.629478,0.483821,0.510025,0.611130,0.254636,0.401692,...,0.50,0.0,0.000000,0.933013,0.75,0.615957,5.840301e-17,0.855832,0.851261,0.668176
1,0.622103,0.651768,0.428597,0.649425,0.615450,0.461053,0.504517,0.606686,0.244847,0.396737,...,0.50,0.0,0.000000,0.933013,0.75,0.000000,0.000000e+00,0.861825,0.845084,0.678400
2,0.609635,0.661344,0.515355,0.633333,0.722369,0.308127,0.664537,0.710014,0.352976,0.603418,...,0.50,0.0,0.333333,0.933013,0.25,0.000000,0.000000e+00,0.974185,0.341392,0.660224
3,0.620342,0.652224,0.405926,0.614943,0.573369,0.392749,0.487992,0.593353,0.215479,0.381872,...,0.50,0.0,0.000000,0.933013,0.75,0.307979,1.000000e+00,0.879156,0.825946,0.709073
4,0.619755,0.652376,0.398369,0.603448,0.559341,0.369981,0.482483,0.588909,0.205690,0.376918,...,0.50,0.0,0.000000,0.933013,0.75,0.862937,8.019377e-01,0.884711,0.819371,0.701120
5,0.619168,0.652528,0.390811,0.591954,0.545314,0.347213,0.476975,0.584465,0.195901,0.371963,...,0.50,0.0,0.000000,0.933013,0.75,1.000000,3.568959e-01,0.890152,0.812702,0.690896
6,0.613856,0.662256,0.533986,0.633333,0.749489,0.275114,0.552716,0.710014,0.360074,0.471224,...,0.75,0.0,0.333333,0.933013,0.25,0.615957,5.840301e-17,0.955307,0.293348,0.648864
7,0.618581,0.652680,0.383254,0.580460,0.531287,0.324445,0.471466,0.580021,0.186112,0.367008,...,0.75,0.0,0.000000,0.933013,0.75,0.615957,5.840301e-17,0.895476,0.805940,0.693168
8,0.617993,0.652832,0.375697,0.568966,0.517260,0.301676,0.465958,0.575576,0.176323,0.362053,...,0.75,0.0,0.000000,0.933013,0.75,0.000000,0.000000e+00,0.900684,0.799087,0.676128
9,0.616232,0.653288,0.353026,0.534483,0.475178,0.233372,0.449433,0.562244,0.146955,0.347189,...,0.75,0.0,0.000000,0.933013,0.75,0.307979,1.000000e+00,0.915589,0.778005,0.673856


In [17]:
train_test_set.get_test_window().shape

(30, 221)

## Setup model

In [18]:
lstm_model_config = ModelConfigDto(
    epochs=5,
    learning_rate=0.001,
    batch_size=32,
)

my_lstm_model = LSTM_Model(
    logger=my_logger, train_test_set=train_test_set, model_config=lstm_model_config
)

PyTorch version: 2.9.0+cpu
Using device: cpu


## Start training the model

In [19]:
my_lstm_model.train()

Training on cpu for 5 epochs...



Epoch 1/5:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch [1/5] - Avg Train Loss: 0.313788


Epoch 2/5:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch [2/5] - Avg Train Loss: 0.257429


...

Epoch [5/5] - Avg Train Loss: 0.086565


Evaluating: 0it [00:00, ?it/s]


✅ Test MSE: nan


d:\GIT\master-thesis\mt_env\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\GIT\master-thesis\mt_env\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
d:\GIT\master-thesis\mt_env\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\GIT\master-thesis\mt_env\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


LSTMForecastModel(
  (lstm): LSTM(220, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=30, bias=True)
  )
)